<a href="https://colab.research.google.com/github/Thienluong9999/IoT_ph-ng_kh-m_th-_y/blob/main/MLTrain_Notebook/Notebook/ML/ReviewData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#review data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# =====================================================
# 1. Đọc Google Sheet
# =====================================================

#online
SHEET_ID = "1kzYAxH2W3ia5sU__4ycZNgGVxvj1bSBiXHULtjBmhQo"
GID = "0"

url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"

df = pd.read_csv(
    url,
    skiprows=1,
    usecols=range(8),
    low_memory=False
)


#offline
# FILE_PATH = r"D:\IoT_Vet_Air_Quality_System\MLTrain_Notebook\datasets\validation\TD1_ESP32_SMELL_Part1.xlsx"

# # đọc bản gốc
# df_raw = pd.read_excel(
#     FILE_PATH,
#     skiprows=1,
#     usecols=range(8)
# )

# tạo bản làm việc
df = df_raw.copy()

# Tạo bản làm việc
df = df_raw.copy()

df.columns = [
    "Date",
    "Time",
    "Status",
    "Temperature",
    "Humidity",
    "MQ2",
    "MQ3",
    "MQ4"
]

# =====================================================
# 2. Ghép Timestamp
# =====================================================

df["Timestamp"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S"
)

df.drop(columns=["Date","Time"], inplace=True)

# =====================================================
# 3. Chuyển kiểu dữ liệu
# =====================================================

for col in ["Temperature","Humidity","MQ2","MQ3","MQ4"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("="*60)
print("INFO")
print("="*60)
print(df.info())

# =====================================================
# 4. Missing Values
# =====================================================

print("\n========== Missing Values ==========")
print(df.isnull().sum())

# =====================================================
# 5. Sampling Interval
# =====================================================

delta = df["Timestamp"].diff()

print("\n========== Sampling Interval ==========")
print(delta.value_counts().head(10))

print("\nMax interval:", delta.max())
print("Min interval:", delta.min())

# khoảng lớn hơn 30 giây
gap = df.loc[delta > pd.Timedelta(seconds=30)]

print("\nGap >30s:", len(gap))

if len(gap) > 0:
    print(gap[["Timestamp"]].head())

# =====================================================
# 6. Physical Range Check
# =====================================================

print("\n========== Physical Range ==========")

print("Temperature:")
print(df[(df["Temperature"]<0)|(df["Temperature"]>50)].shape[0])

print("Humidity:")
print(df[(df["Humidity"]<0)|(df["Humidity"]>100)].shape[0])

for col in ["MQ2","MQ3","MQ4"]:
    print(col,
          df[(df[col]<0)|(df[col]>4095)].shape[0])

# =====================================================
# 7. Statistics
# =====================================================

print("\n========== Statistics ==========")
print(df.describe())

# =====================================================
# 8. Quantiles
# =====================================================

print("\n========== Quantiles ==========")

for col in ["MQ2","MQ3","MQ4"]:

    print("\n",col)

    print(df[col].quantile([
        0.90,
        0.95,
        0.99,
        0.995,
        0.999
    ]))

# =====================================================
# 9. IQR Outlier
# =====================================================

print("\n========== IQR Outlier ==========")

for col in ["MQ2","MQ3","MQ4"]:

    q1=df[col].quantile(.25)
    q3=df[col].quantile(.75)

    iqr=q3-q1

    lower=q1-1.5*iqr
    upper=q3+1.5*iqr

    out=df[(df[col]<lower)|(df[col]>upper)]

    print(f"\n{col}")
    print("Lower =",lower)
    print("Upper =",upper)
    print("Outlier =",len(out))
    print("Percent =",100*len(out)/len(df))

# =====================================================
# 10. Boxplot
# =====================================================

plt.figure(figsize=(10,5))

plt.boxplot(
    [
        df["MQ2"],
        df["MQ3"],
        df["MQ4"]
    ],
    labels=["MQ2","MQ3","MQ4"]
)

plt.title("Boxplot")
plt.grid(True)

plt.show()

# =====================================================
# 11. Time Series
# =====================================================

for col in ["MQ2","MQ3","MQ4"]:

    plt.figure(figsize=(18,4))

    plt.plot(
        df["Timestamp"],
        df[col],
        linewidth=.5
    )

    plt.title(col)
    plt.xlabel("Time")
    plt.ylabel("ADC")

    plt.grid(True)

    plt.show()

# =====================================================
# 12. Histogram
# =====================================================

for col in ["MQ2","MQ3","MQ4"]:

    plt.figure(figsize=(7,4))

    plt.hist(
        df[col],
        bins=50
    )

    plt.title(col)

    plt.xlabel("ADC")
    plt.ylabel("Count")

    plt.grid(True)

    plt.show()

print("\nDONE")